# 🎯 AI Product Recommendation Engine — IoT Edition
Recommandations logiques pour une boutique de matériel IoT/électronique.

In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
import re, unicodedata, warnings
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")
print("✅ Dépendances installées!")

## 📊 Étape 1 : Chargement des données

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("cothings-ai/inventory_export.csv"),
    Path("data/inventory_export.csv")
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"✅ {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## 🧹 Étape 2 : Nettoyage + Catégorisation IoT

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par"}
    return " ".join(t for t in text.split() if len(t)>2 and t not in stop and not t.isdigit())

# Catégories IoT enrichies
CATEGORY_RULES = {
    "batterie":    ["pile","piles","batterie","battery","18650","lipo","lithium","alkaline","r20","r6","aa","aaa","4ah"],
    "chargeur":    ["charge","charging","chargeur","chargement","usb","port","bouclier"],
    "led":         ["led","ruban","lampe","lumiere","ir","850nm","spectre","rgb"],
    "alimentation":["pwm","regulateur","regulator","boost","dc","7805","79l12","convertisseur","tension","vitesse","moteur"],
    "electronique":["module","hdmi","circuit","74hc","potentiometer","adaptateur","registre","afficheur","lcd","i2c","schmitt"],
    "connectique": ["cable","cosses","pinces","connecteur","crocodile","jumper","cavalier","barette","fil"],
    "raspberry":   ["raspberry","rpi","camera","vision","nocturne","microbit"],
    "mobilite":    ["voiture","telecommandee","rc","drone","helices","roue","chassis","whl","4wd","mcnamum"],
    "composant":   ["diode","resistance","resistances","1n4148","tube","thermoretractable","bague","spiral","support"],
    "outillage":   ["cutter","lame","lames","scie","outil","percage","hssd"],
    "rf":          ["433mhz","433","rf","stx","srx","recepteur","emetteur","bluetooth","soundtech"],
}

def infer_category(title):
    n = normalize_text(title)
    for cat, kws in CATEGORY_RULES.items():
        if any(k in n for k in kws):
            return cat
    return "autre"

# === CARTE DE COMPLÉMENTARITÉ IoT ===
# Clé = catégorie du produit, Valeur = catégories recommandées (par priorité)
COMPLEMENTARITY_MAP = {
    "batterie":     ["chargeur", "alimentation", "composant", "connectique"],
    "chargeur":     ["batterie", "connectique", "electronique"],
    "led":          ["alimentation", "connectique", "electronique", "raspberry"],
    "alimentation": ["electronique", "connectique", "led", "batterie"],
    "electronique": ["connectique", "alimentation", "composant", "raspberry"],
    "connectique":  ["electronique", "alimentation", "composant"],
    "raspberry":    ["led", "electronique", "connectique", "alimentation", "camera"],
    "mobilite":     ["batterie", "chargeur", "rf", "alimentation"],
    "composant":    ["electronique", "connectique", "alimentation"],
    "outillage":    ["composant", "connectique", "electronique"],
    "rf":           ["alimentation", "electronique", "connectique", "mobilite"],
    "autre":        ["electronique", "connectique", "composant"],
}

df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""
df["location"]       = df["location"].astype(str).str.strip() if "location" in df.columns else "unknown"

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["category"]    = df["product_title"].apply(infer_category)
df["clean_title"] = df["product_title"].apply(normalize_text)

print("✅ Nettoyage terminé!")
print("
📊 Distribution des catégories:")
print(df.drop_duplicates("product_handle")["category"].value_counts())

## 📈 Étape 3 : Agrégation des produits

In [ ]:
products = df.drop_duplicates(subset=["product_handle"]).copy()

inv = df.groupby("product_handle").agg({
    "available (not editable)":"sum",
    "on hand (current)":"sum",
    "on hand (new)":"sum"
}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]

products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = (
    "produit " + products["product_title"].fillna("") + " " +
    "categorie " + products["category"].fillna("") + " " +
    "motscles " + products["clean_title"].fillna("")
)
products = products[["product_handle","product_title","category","sku","product_profile","available_total","on_hand_current","on_hand_new"]].reset_index(drop=True)

print(f"✅ {len(products)} produits uniques")
print(f"   En stock: {len(products[products['available_total']>0])}")
products.head(5)

## 🤖 Étape 4 : Moteur de recommandation IoT

In [ ]:
class IoTRecommendationEngine:
    """
    Moteur de recommandation logique pour boutique IoT/électronique.
    Priorité: complémentarité métier > similarité sémantique.
    """

    def __init__(self, products_df, complementarity_map):
        self.products = products_df.copy().reset_index(drop=True)
        self.comp_map = complementarity_map

        # Préparation texte
        self.products["clean_profile"] = self.products["product_profile"].fillna("").apply(self._normalize)
        self.products["keyword_set"]   = self.products["clean_profile"].apply(lambda x: set(x.split()))

        # Score de stock normalisé
        max_stock = self.products["available_total"].max()
        self.products["stock_score"] = (self.products["available_total"] / max(max_stock,1)).clip(0,1)

        # Embeddings sémantiques
        print("🔄 Chargement du modèle sémantique...")
        self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        texts = [f"passage: {p}" for p in self.products["clean_profile"]]
        self.embeddings = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        self.nn = NearestNeighbors(metric="cosine", algorithm="brute")
        self.nn.fit(self.embeddings)
        print(f"✅ Modèle prêt — {len(self.products)} produits indexés")

    def _normalize(self, text):
        text = "" if pd.isna(text) else str(text)
        text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
        text = re.sub(r"[^a-z0-9\s]"," ",text)
        stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec"}
        return " ".join(t for t in text.split() if len(t)>2 and t not in stop and not t.isdigit())

    def get_product_index(self, product_title):
        if isinstance(product_title, int):
            return product_title if 0 <= product_title < len(self.products) else None
        title_lower = str(product_title).lower().strip()
        matches = self.products[self.products["product_title"].str.lower().str.contains(title_lower, na=False)]
        return int(matches.index[0]) if len(matches) > 0 else None

    def _complementarity_score(self, source_category, candidate_category):
        """Score 0-1 basé sur la carte de complémentarité métier."""
        if source_category == candidate_category:
            return -0.3   # Pénalité doublons (même catégorie)
        complementary = self.comp_map.get(source_category, [])
        if candidate_category in complementary:
            rank = complementary.index(candidate_category)
            return 0.8 - (rank * 0.1)   # 0.8, 0.7, 0.6, 0.5 selon rang
        return 0.0

    def _keyword_overlap(self, set_a, set_b):
        if not set_a or not set_b:
            return 0.0
        return len(set_a & set_b) / max(1, len(set_a | set_b))

    def recommend(self, product_titles, n_recommendations=5, in_stock_only=True):
        """
        Recommande des produits complémentaires (logique IoT).
        
        Scoring final:
          40% complémentarité métier (catégorie)
          35% similarité sémantique (embeddings)
          15% overlap de mots-clés
          10% score de stock
        """
        if isinstance(product_titles, (str, int)):
            product_titles = [product_titles]

        source_indices = [self.get_product_index(t) for t in product_titles]
        source_indices = [i for i in source_indices if i is not None]
        if not source_indices:
            print("❌ Produit(s) introuvable(s)")
            return pd.DataFrame()

        source_cats    = set(self.products.loc[source_indices, "category"].tolist())
        source_kws     = set().union(*self.products.loc[source_indices, "keyword_set"].tolist())
        source_emb     = np.mean(self.embeddings[source_indices], axis=0)
        source_emb    /= (np.linalg.norm(source_emb) + 1e-8)

        # Similarités sémantiques via NearestNeighbors
        distances, indices = self.nn.kneighbors([source_emb], n_neighbors=min(len(self.products), 40))
        sem_score_map = {}
        for d, i in zip(distances[0], indices[0]):
            sem_score_map[i] = 1 - float(d)

        scores = []
        for idx, row in self.products.iterrows():
            if idx in source_indices:
                continue
            if in_stock_only and row["available_total"] <= 0:
                continue

            # Score complémentarité (max sur toutes les catégories sources)
            comp_s = max(self._complementarity_score(sc, row["category"]) for sc in source_cats)

            # Score sémantique
            sem_s = sem_score_map.get(idx, 0.0)

            # Score overlap mots-clés
            kw_s = self._keyword_overlap(source_kws, row["keyword_set"])

            # Score stock
            stk_s = float(row["stock_score"])

            final = 0.40*comp_s + 0.35*sem_s + 0.15*kw_s + 0.10*stk_s

            # Déterminer le type de match
            if comp_s >= 0.5:
                match_type = "🔗 Complémentaire"
            elif comp_s >= 0.0:
                match_type = "🔄 Associé"
            elif comp_s < 0:
                match_type = "⚠️ Même catégorie"
            else:
                match_type = "📎 Sémantique"

            scores.append({"idx": idx, "score": final, "comp": comp_s, "sem": sem_s, "match_type": match_type})

        if not scores:
            print("⚠️ Aucune recommandation trouvée.")
            return pd.DataFrame()

        scores_df = pd.DataFrame(scores).sort_values("score", ascending=False).head(n_recommendations)

        result = self.products.loc[scores_df["idx"]].copy()
        result["recommendation_score"] = scores_df["score"].values
        result["complementarity"]       = scores_df["comp"].values
        result["semantic_similarity"]   = scores_df["sem"].values
        result["match_type"]            = scores_df["match_type"].values
        result["in_stock"]              = result["available_total"] > 0
        return result.reset_index(drop=True)


engine = IoTRecommendationEngine(products, COMPLEMENTARITY_MAP)

## ⭐ Étape 5 : Test — Recommandation produit unique

In [ ]:
in_stock = products[products["available_total"] > 0]

def show_recommendations(product_titles, n=5, in_stock_only=True):
    if isinstance(product_titles, str):
        product_titles = [product_titles]
    
    print(f"
🎯 Produit(s) sélectionné(s):")
    for t in product_titles:
        idx = engine.get_product_index(t)
        if idx is not None:
            row = products.loc[idx]
            print(f"   • {row['product_title']} [{row['category']}] — Stock: {row['available_total']}")
    
    recs = engine.recommend(product_titles, n_recommendations=n, in_stock_only=in_stock_only)
    
    if len(recs) == 0:
        print("Aucun résultat.")
        return
    
    print(f"
💡 Top {n} Recommandations:
")
    print(f"{'#':<3} {'Produit':<55} {'Catégorie':<14} {'Type':<20} {'Score':>6} {'Stock':>6}")
    print("-"*115)
    for i, (_, row) in enumerate(recs.iterrows(), 1):
        print(f"{i:<3} {row['product_title'][:54]:<55} {row['category']:<14} {row['match_type']:<20} {row['recommendation_score']:>5.0%} {row['available_total']:>6}")

# Test 1 : Batterie → doit recommander chargeur + alimentation
show_recommendations("18650 3.7V 2600MAH Batterie")

In [ ]:
# Test 2 : LED → doit recommander alimentation + connectique
show_recommendations("Ruban LED")

In [ ]:
# Test 3 : Bundle — Drone + Batterie → doit recommander chargeur + RF
show_recommendations(["Hélices", "18650 3.7V 2600MAH Batterie"], n=6)

In [ ]:
# Test 4 : Raspberry Pi → doit recommander LED IR, câbles, alimentation
show_recommendations("Raspberry")

## 🔧 Étape 6 : Personnaliser la carte de complémentarité

Pour améliorer les résultats, édite ce dictionnaire selon tes connaissances métier :

In [ ]:
# Ajoute ici tes règles personnalisées
# Format: "categorie_source": ["categorie_priorité1", "categorie_priorité2", ...]
COMPLEMENTARITY_MAP["led"] = ["alimentation", "connectique", "electronique", "raspberry"]
COMPLEMENTARITY_MAP["mobilite"] = ["batterie", "chargeur", "rf", "alimentation", "electronique"]

# Recharger le moteur avec la nouvelle carte
engine.comp_map = COMPLEMENTARITY_MAP
print("✅ Carte de complémentarité mise à jour!")
show_recommendations("Voiture Télécommandée")